In [1]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import requests

SEED = 6
np.random.seed(SEED)

ROOT = Path(".")
DATA_DIR = ROOT / "data"
EXT_DIR  = ROOT / "external" / "replogle_raw"
EXT_DIR.mkdir(parents=True, exist_ok=True)

MEANS_PATH = DATA_DIR / "training_data_means.csv"
VALMAP_PATH = DATA_DIR / "pert_ids_val.csv"
H5AD_INTERNAL_PATH = DATA_DIR / "training_cells.h5ad"

df_means = pd.read_csv(MEANS_PATH)
df_valmap = pd.read_csv(VALMAP_PATH)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]
baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_perts = df_train["pert_symbol"].astype(str).to_numpy()
Y_means = (df_train[gene_columns].to_numpy(np.float32) - x_base[None, :])  # (80, 5127)

val_targets = df_valmap["pert"].astype(str).tolist()

print("Y_means:", Y_means.shape, "G:", len(gene_columns))

Y_means: (80, 5127) G: 5127


In [2]:
FIGSHARE_ARTICLE_ID = 20029387

def figshare_list_files(article_id: int):
    url = f"https://api.figshare.com/v2/articles/{article_id}/files"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    files = r.json()
    rows = []
    for f in files:
        rows.append({
            "name": f.get("name"),
            "id": f.get("id"),
            "size_GB": round(float(f.get("size", 0)) / (1024**3), 3),
        })
    return pd.DataFrame(rows).sort_values("size_GB", ascending=False), files

df_files, raw_files = figshare_list_files(FIGSHARE_ARTICLE_ID)
df_files.head(30)

,name,id,size_GB
6,K562_gwps_raw_singlecell_01.h5ad,35775507,61.310
4,K562_gwps_normalized_singlecell_01.h5ad,35774440,61.310
3,K562_essential_raw_singlecell_01.h5ad,35773219,9.930
1,K562_essential_normalized_singlecell_01.h5ad,35773075,9.930
8,rpe1_normalized_singlecell_01.h5ad,35775554,8.103
5,K562_gwps_raw_bulk_01.h5ad,35774443,0.349
2,K562_gwps_normalized_bulk_01.h5ad,35773217,0.349
7,rpe1_normalized_bulk_01.h5ad,35775512,0.089
9,rpe1_raw_bulk_01.h5ad,35775581,0.089
0,K562_essential_raw_bulk_01.h5ad,35773070,0.074


In [14]:
FILE_NAME = "K562_gwps_raw_singlecell_01.h5ad"  # then switch to K562_gwps_raw_singlecell_01.h5ad

row = df_files[df_files["name"] == FILE_NAME]
assert len(row) == 1
FILE_ID = int(row.iloc[0]["id"])

EXT_H5AD_PATH = EXT_DIR / FILE_NAME
print("Selected:", FILE_NAME, "FILE_ID:", FILE_ID)
print("Path:", EXT_H5AD_PATH)

Selected: K562_gwps_raw_singlecell_01.h5ad FILE_ID: 35775507
Path: external\replogle_raw\K562_gwps_raw_singlecell_01.h5ad


In [15]:
def download_figshare_file(file_id: int, out_path: Path, chunk_bytes: int = 1024 * 1024 * 16):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        print("[dl] exists:", out_path, out_path.stat().st_size)
        return

    url = f"https://api.figshare.com/v2/file/download/{file_id}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_bytes):
                if chunk:
                    f.write(chunk)
    print("[dl] done:", out_path, out_path.stat().st_size)

download_figshare_file(FILE_ID, EXT_H5AD_PATH)

[dl] done: external\replogle_raw\K562_gwps_raw_singlecell_01.h5ad 65830941948


In [27]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad("external/replogle_raw/K562_gwps_raw_singlecell_01.h5ad", backed="r")
print("shape:", adata.n_obs, adata.n_vars)

geneU = np.array([str(g).upper() for g in gene_columns], dtype="U")

# 1) var_names overlap (works if var_names are symbols)
varU = np.char.upper(np.asarray(adata.var_names.astype(str)).astype("U"))
print("overlap using var_names:", len(set(varU).intersection(set(geneU))), "/ 5127")

# 2) if gene_name exists, overlap using gene_name (works if var_names are ENSG)
if "gene_name" in adata.var.columns:
    symU = np.char.upper(np.asarray(adata.var["gene_name"].astype(str)).astype("U"))
    print("overlap using var['gene_name']:", len(set(symU).intersection(set(geneU))), "/ 5127")
else:
    print("no var['gene_name'] column")

shape: 1989578 8248
overlap using var_names: 0 / 5127
overlap using var['gene_name']: 2385 / 5127


In [ ]:
import numpy as np
import torch

# ---------- Build ext_raw / ext_pertsU if you only have dict ----------
if "ext_delta_by_pertU" in globals() and ("ext_raw" not in globals() or ext_raw is None):
    ext_pertsU = sorted(ext_delta_by_pertU.keys())
    ext_raw = np.vstack([ext_delta_by_pertU[p] for p in ext_pertsU]).astype(np.float32)
    print("[ext] built ext_raw from dict:", ext_raw.shape)

assert "ext_meta" in globals() and ("present_pos" in ext_meta), "Need ext_meta['present_pos'] from the external builder."
present_pos = np.asarray(ext_meta["present_pos"], dtype=np.int64)
print("[ext] present_pos:", len(present_pos), "/ 5127")

def complete_ext_deltas_via_uout(ext_raw, present_pos, U_out, ridge_l2=1e-2, clip=5.0):
    """
    ext_raw: (N_ext, 5127) with valid values only on present_pos (others can be 0)
    present_pos: indices where ext_raw contains real measured values
    U_out: (5127, k) gene basis (your control-SVD U_out is perfect)
    returns ext_full: (N_ext, 5127)
    """
    ext_raw = ext_raw.astype(np.float32, copy=False)
    U = U_out.astype(np.float32, copy=False)
    Up = U[present_pos, :]  # (Gp, k)

    # Dp = measured deltas (N_ext, Gp)
    Dp = ext_raw[:, present_pos].astype(np.float32)

    k = U.shape[1]
    Gk = (Up.T @ Up).astype(np.float32) + float(ridge_l2) * np.eye(k, dtype=np.float32)
    A = np.linalg.solve(Gk, Up.T).astype(np.float32)  # (k, Gp)

    coeffs = (Dp @ A.T).astype(np.float32)            # (N_ext, k)
    ext_full = (coeffs @ U.T).astype(np.float32)      # (N_ext, 5127)

    if clip is not None:
        ext_full = np.clip(ext_full, -float(clip), float(clip)).astype(np.float32)

    return ext_full

# ---- complete targets ----
EXT_COMPLETE_L2 = 1e-2
EXT_COMPLETE_CLIP = 5.0

ext_full = complete_ext_deltas_via_uout(ext_raw, present_pos, U_out, ridge_l2=EXT_COMPLETE_L2, clip=EXT_COMPLETE_CLIP)
print("[ext] ext_full:", ext_full.shape, "absmax p99:", np.percentile(np.max(np.abs(ext_full), axis=1), 99))

# ---- build Z_ext embeddings ----
Z_ext = np.vstack([emb_pert(p) for p in ext_pertsU]).astype(np.float32)
print("[ext] Z_ext:", Z_ext.shape)

# torch tensors
Z_ext_t = torch.tensor(Z_ext, device=device, dtype=torch.float32)
ext_full_t = torch.tensor(ext_full, device=device, dtype=torch.float32)

,gene_name,chr,start,end,class,strand,length,in_matrix,mean,std,cv,fano
gene_id,,,,,,,,,,,,
ENSG00000237491,LINC01409,chr1,778747,810065,gene_version10,+,31318,True,0.116626,0.349971,3.000803,1.050194
ENSG00000228794,LINC01128,chr1,825138,868202,gene_version9,+,43064,True,0.182850,0.437274,2.391434,1.045713
ENSG00000188976,NOC2L,chr1,944203,959309,gene_version11,-,15106,True,1.415674,1.397208,0.986957,1.378984
ENSG00000187961,KLHL17,chr1,960584,965719,gene_version14,+,5135,True,0.105599,0.330678,3.131439,1.035497
ENSG00000188290,HES4,chr1,998962,1000172,gene_version10,-,1210,True,0.242700,0.550596,2.268630,1.249098
...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000198727,MT-CYB,chrM,14747,15887,gene_version2,+,1140,True,116.549202,59.102600,0.507104,29.971182
ENSG00000278704,BX004987.1,GL000009.2,56140,58376,gene_version1,-,2236,True,0.195008,0.455099,2.333745,1.062085
ENSG00000278384,AL354822.1,GL000218.1,51867,54893,gene_version1,-,3026,True,0.190869,0.450850,2.362089,1.064948
